# **Reward Modeling with GPT-2**

Imagine you're working as a machine learning engineer at a healthcare technology company that wants to integrate advanced language models into its medical assistant platform. Your task is to evaluate and select the best large language model (LLM) that can accurately interpret patient queries, assist doctors with clinical documentation, and generate clear, empathetic, and medically sound responses.

However, simply deploying a powerful LLM isn't enough. A general-purpose model might give technically correct answers while being too cold in tone, overly verbose, or even potentially misleading in a medical context. This is where reward models become essential. By training a reward model, you can teach the LLM to prioritize the right behaviors — such as being concise, compassionate, and clinically accurate — ensuring its responses align with both medical best practices and the company's commitment to patient safety.

In this hands-on lab, you dive into the process of building and training a reward model using the Transformer Reinforcement Learner (TRL) library from Hugging Face. You'll learn how to configure the training environment, define meaningful reward signals that reflect what a good medical response looks like, and fine-tune the LLM to consistently produce high-quality, context-aware output. By the end, you'll have practical experience applying reward modeling to a real-world domain — giving you the tools to shape AI behavior in any high-stakes application.

### **Import Libraries**

In [138]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datasets import load_dataset
from datasets import DatasetDict
from transformers import AutoTokenizer, GPT2ForSequenceClassification, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model, get_peft_model
from trl import RewardTrainer,RewardConfig


In [75]:
dataset_name = "Dahoas/synthetic-instruct-gptj-pairwise"

In [76]:
dataset = load_dataset(dataset_name)

In [77]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 33143
    })
})

In [131]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu" )
device.type

'cuda'

In [141]:
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=1)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4624.96it/s]
[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [143]:
model.to(device)

GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=1, bias=False)
)

In [144]:
# Add special tokens if necessary
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

max_length = 1024

In [145]:
dataset["train"][0]

{'prompt': 'I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.',
 'chosen': "Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.",
 'rejected': 'H

**Need to filter the chosen and rejected responses with the prompt (which user's input). To do that we are going to use a Lamda Function with the relevant respose type**

In [146]:
get_res = lambda dataset, res_type: ["\n\nHuman: " + prompt + "\n\nAssistant: "+ response 
                                    for prompt, response in zip(dataset['train']['prompt'], dataset['train'][res_type])]

In [147]:
res_chosen = get_res(dataset, 'chosen') 
res_rejected = get_res(dataset, 'rejected')

In [148]:
for i in range (6):
    print ("response chosen:", res_chosen[i])



response chosen: 

Human: I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.

Assistant: Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.
respo

In [ ]:
for i in range (6):
    print ("response rejected:", res_rejected[i])

response rejected: 

Human: I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.

Assistant: How do I store a bagels for eating at a later date?


 You can place the bagels in an airtight container and reheat them in the microwave.  Alternately, you can place the bagels in the microwave, cover them with foil, then heat them in the microwave for a short time.
response rejected: 

Human: What type of wine goes best with steak.

Assistant: Choose leaner cuts of steak or a burger dish, because the flavor of wine is the most versatile.  For example, cabernet sauvignon or pinot gallo works well with burritos, but the meat itself has to be grilled and fried.  
2.  Pour a good quality dry white wine that’s moderately young.  Chardonnay or far-lauter is also nice for steak.  
3.  Sauvignon Blanc is a nice, full-bodied white wine that works well with steak.
4.  Chenin Blanc is a grape that grows in northwest France, so it often tastes a bit strong

**It is more helpful when we have seperate columns to define prompt with chosen and prmopt with rejected.**

In [150]:
def add_seperate_res_column(mydataset):
    
    mydataset['prompt_chosen'] = "\n\nHuman: "+ mydataset['prompt'] + "\n\nAssistant: "+ mydataset['chosen']
    mydataset['prompt_rejected'] = "\n\nHuman" + mydataset['prompt'] + "\n\nAssistant: " + mydataset['rejected']
    
    return mydataset
    

In [151]:
dataset['train'] = dataset['train'].map(add_seperate_res_column)

Map: 100%|██████████| 33043/33043 [00:05<00:00, 6549.41 examples/s]


In [152]:
dataset['train'][0]

{'prompt': 'I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.',
 'chosen': "Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.",
 'rejected': 'H

**Identifyting the maximum sample length in the dataset**

In [153]:
get_max_length = lambda data_list: max([len(sample) for sample in data_list])

In [154]:
max_len_chosen = get_max_length(res_chosen)
max_len_rejected = get_max_length(res_rejected)

print(f"max length of res_chosen:{max_len_chosen}")
print(f"max length of res_rejected:{max_len_rejected}")

max length of res_chosen:3167
max length of res_rejected:5011


**We need to filter the datset based on the model's max length (=1024). So if we get data that beyond the model's max length, the row data will be truncate and there have a chance to miss the important information that valuable to train the model.**

In [155]:
find_ids = lambda mydataset, max_len: [
    i for i, (chosen,rejected) in enumerate(zip(mydataset['prompt_chosen'], mydataset['prompt_rejected']))
    if len(chosen) < max_len or len(rejected)< max_len
]

In [156]:
selected_ids = find_ids(dataset['train'], max_length)


In [157]:
selected_ids[0:10]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [158]:
dataset['train'] = dataset['train'].select(selected_ids)


In [159]:
dataset['train'][0]

{'prompt': 'I was wondering if you could walk me through the process of setting up a hydroponic garden for herbs.',
 'chosen': "Sure! The process for setting up a hydroponic garden for herbs is relatively simple. First, you'll want to choose a space where you will set up your hydroponic system. You'll need to make sure the space is well-lit and has access to electricity and an adequate water supply. Next, you'll need to choose the type of hydroponic system you want to use. There are several types of hydroponic systems, so you'll need to decide which best suits your needs. Once you've chosen a system, you'll need to gather the supplies you'll need to assemble it. This includes things like pumps, growing trays, grow lights, and nutrients. Once you've assembled the system, you'll need to add your choice of herbs to the system. Lastly, you'll need to monitor and adjust the system as needed to ensure your herbs are getting the correct amount of light, water, and nutrients.",
 'rejected': 'H

In [184]:
split_original_dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)


In [185]:
dataset_original_dict = DatasetDict({
    "train": split_original_dataset['train'],
    "test": split_original_dataset['test']
})

### **Data Tokenization Function**

The preprocess_function converts the prompt_chosen and prompt_rejected fields into tokens, which the RewardTrainer needs to function properly. Think of chosen as the "good answer" and rejected as the "bad answer."
By tokenizing both, the model can numerically compare what makes one response better than the other. Feeding the trainer both examples side by side is what allows it to learn the difference between a high-quality and a low-quality output — which is the whole point of reward-based training: teaching the model to consistently favor better responses.

In [160]:
def data_tokenization (mydataset):
    
    tokenized_chosen = tokenizer(mydataset['prompt_chosen'], 
                                max_length=max_length, padding='max_length', truncation=True)

    tokenized_rejected = tokenizer(mydataset['prompt_rejected'], max_length= max_length, 
                                padding = 'max_length', truncation=True)
    
    return {
        "input_ids_chosen": tokenized_chosen['input_ids'],
        "attention_mask_chosen": tokenized_chosen['attention_mask'],
        "input_ids_rejected": tokenized_rejected['input_ids'],
        "attention_mask_rejected": tokenized_rejected['attention_mask']
    }

#### Create a Dictionary to Access Chosen and Rejected Prompt for Model Validation

In [173]:
train_str = {"chosen": [data for data in dataset['train']['prompt_chosen']], "rejected": [data for data in dataset['train']['prompt_rejected']]}

Map the tokenized function with dataset:

In [162]:
dataset_tokenized:DatasetDict = {'train': {}}

In [ ]:
dataset_tokenized = DatasetDict({
    "train": dataset['train'].map(data_tokenization, batched=True,
            remove_columns=['prompt',"chosen", "rejected",'prompt_chosen', 'prompt_rejected'])
})

Map: 100%|██████████| 33043/33043 [00:27<00:00, 1185.22 examples/s]


In [164]:
dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected'],
        num_rows: 33043
    })
})

Split the tokenized data set into train and test partitions

In [175]:
dataset_split = dataset_tokenized['train'].train_test_split(test_size=0.2, seed=42)

In [176]:
dataset_dict = DatasetDict({
    "train": dataset_split['train'],
    "test": dataset_split['test'],
})

In [177]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected'],
        num_rows: 26434
    })
    test: Dataset({
        features: ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected'],
        num_rows: 6609
    })
})

### **LoRA Configuration**

With the training dataset ready, the next step is fine-tuning the pretrained model. Rather than updating every single parameter in the model — which is computationally expensive — it's smarter to use **LoRA (Low-Rank Adaptation)**, which only modifies a small, targeted subset of the model's weights.

To set this up, you create a `LoraConfig` object from the `peft` library with the following settings:

- **`task_type=TaskType.SEQ_CLS`** — Tells LoRA that this is a sequence classification job, so it configures itself appropriately for that task.
- **`inference_mode=False`** — Puts the model in training mode, meaning weights will be updated during the process.
- **`r=8`** — Controls how "compressed" the LoRA matrices are. A lower rank means fewer parameters to train, keeping things lightweight.
- **`lora_alpha=32`** — Acts as a scaling factor that controls how much influence the LoRA updates have on the original model weights.
- **`lora_dropout=0.1`** — Randomly drops 10% of LoRA layer connections during training to reduce the risk of overfitting.
- **`target_modules=["attn.c_attn", "attn.c_proj"]`** — Pins LoRA specifically to the attention layers of the model, since these are the most impactful parts to adapt for instruction-following tasks.

The beauty of this approach is that instead of retraining the entire model from scratch, you're surgically fine-tuning only the parts that matter most — making the process faster, cheaper, and just as effective.

> **Key takeaway:** LoRA lets you get the benefits of full fine-tuning at a fraction of the cost.

In [178]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['attn.c_attn', 'attn.c_proj']
)

Let's Define the Peft Model

In [179]:
peft_model = get_peft_model(model, lora_config)

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\peft\tuners\tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [180]:
peft_model.to(device)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): GPT2ForSequenceClassification(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): P

Define TrainingArguments

In [181]:
training_args = RewardConfig(
    output_dir="./model_output",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    gradient_accumulation_steps=8,
    logging_steps=10,
    eval_strategy="steps",
    save_strategy="steps",
    learning_rate=1.41e-5,
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True
    
)

Define RewardTrainer

**Newer versions of RewardTrainer we should have to give the raw dataset. RewardTrainer internally tokenized the dataset and process.**

In [ ]:
reward_trainer = RewardTrainer(
    model= peft_model,
    args=training_args,
    train_dataset=dataset_original_dict["train"],
    eval_dataset=dataset_original_dict["test"],
    processing_class=tokenizer
)

Filtering eval >1024 tokens: 100%|██████████| 6609/6609 [00:01<00:00, 5094.11 examples/s]


Model Trainer

In [188]:
reward_trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss


KeyboardInterrupt: 